# Step 02 — Rebuild region-aware ATF thresholds from the 37 files

This notebook reruns the ATF threshold rebuild on the **renamed 37-file dataset**.

## Reuse from `astro_atf_analysis_improved_sectioned.ipynb`

This step **does** reuse/adapt the working logic from the reference notebook through `src.atf_reference_adapter` and `src.atf_step02`:

- ATF discovery
- ATF parsing
- preprocessing / artifact handling
- sweep-level feature extraction

What is new here is the **repository-facing packaging** around that logic:

- region-aware threshold tables,
- redundancy diagnostics,
- region-effect summaries,
- feature reliability weights,
- and condition-level reliability.

## Important notebook display change

This notebook deliberately shows:
- the full canonical sweep-level feature table,
- the full redundancy table,
- the full region-effect table,
- and the full condition-level reliability table.

## Not included here

The Numba-vs-NumPy benchmark is intentionally omitted from step 02.  
That benchmark only becomes meaningful in later optimization-heavy steps.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(os.environ.get("ASTROMODEL_PROJECT_ROOT", Path.cwd())).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.atf_step02 import run_step02_rebuild_atf_thresholds

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 200)

PROJECT_ROOT

In [ ]:
results = run_step02_rebuild_atf_thresholds(PROJECT_ROOT)

feature_df = results["feature_table_by_sweep"]
cell_counts = results["region_condition_cell_counts"]
redundancy = results["redundancy_diagnostics"]
reliability = results["feature_reliability_weights"]
condition_reliability = results["condition_feature_reliability"]
thresholds = results["condition_region_sweep_thresholds"]
region_effects = results["region_effect_summary"]

print("written outputs:", sorted((PROJECT_ROOT / "outputs" / "features").glob("*.csv")))
print("feature rows:", len(feature_df), "unique files:", feature_df["file_id"].nunique())

## Region × condition cell counts

In [ ]:
display(cell_counts)

fig, ax = plt.subplots(figsize=(7, 4))
plot_df = cell_counts.copy()
plot_df["label"] = plot_df["region"] + "_" + plot_df["condition"]
ax.bar(plot_df["label"], plot_df["n_cells"])
ax.set_ylabel("n_cells")
ax.set_title("ATF cell counts by region and condition")
ax.tick_params(axis="x", rotation=45)
plt.show()

## Canonical sweep-level feature table

In [ ]:
feature_view = feature_df[
    [
        "file_id",
        "region",
        "condition",
        "sweep",
        "peak_depolarization_mV",
        "stim_end_depolarization_mV",
        "rise_slope_mV_per_s",
        "rise_tau_s",
        "plateau_slope_mV_per_s",
        "decay_slope_mV_per_s",
        "decay_tau_s",
        "undershoot_magnitude_mV",
        "return_slope_mV_per_s",
        "n_artifact_points_total",
    ]
].copy()
print(feature_view.round(4).to_string(index=False))

## Redundancy diagnostics

In [ ]:
print(redundancy.round(4).to_string(index=False))

## Condition-level feature reliability

In [ ]:
print(condition_reliability.round(4).to_string(index=False))

## Region-specific feature reliability

In [ ]:
print(reliability.round(4).to_string(index=False))

## Region-aware thresholds

In [ ]:
print(thresholds.round(4).to_string(index=False))

## Region-effect summary

In [ ]:
print(region_effects.round(4).to_string(index=False))

## Consequence for later steps

Two practical conclusions stand out:

1. `peak_depolarization_mV` and `stim_end_depolarization_mV` are near-duplicates here and should not both dominate later losses.
2. `return_slope_mV_per_s` has condition-dependent missingness, so its weight should be explicitly moderated rather than treated as uniformly reliable across conditions.